# Phase 5: DPO — Direct Preference Optimization

**Goal**: Improve Tamil response quality using preference learning — no human raters needed.

**Method**: Generate chosen (T=0.3, focused) and rejected (T=1.2, noisy) responses from the SFT model. Train with DPO to prefer the better responses.

**Starts from**: `wickkiey/tamil-llama-3.1-8b-sft-v1`  
**Hardware**: Colab A100 40GB  
**Output**: `wickkiey/tamil-llama-3.1-8b-dpo-v1`

> DPO holds two copies of the model in memory (policy + reference). 40GB is needed.

In [ ]:
import subprocess, sys
for pkg in ["unsloth", "trl>=0.8.6", "datasets", "peft", "accelerate", "bitsandbytes"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("Ready.")

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
SFT_CHECKPOINT  = "wickkiey/tamil-llama-3.1-8b-sft-v1"   # Phase 4 output
MAX_SEQ_LEN     = 2048

# Preference pair generation settings
N_PREF_PAIRS    = 8000        # number of preference pairs to generate
CHOSEN_TEMP     = 0.3         # low temperature → focused, higher quality
REJECTED_TEMP   = 1.2         # high temperature → noisy, lower quality
MAX_GEN_TOKENS  = 250

# DPO training settings
DPO_BETA        = 0.1         # KL penalty coefficient
DPO_LR          = 5e-7        # very low LR for DPO stability
DPO_EPOCHS      = 1
DPO_BATCH       = 2
DPO_GRAD_ACCUM  = 4
LORA_RANK       = 64          # lower rank for DPO (fine adjustment)

OUTPUT_DIR      = "outputs/dpo_v1"
HF_REPO         = "wickkiey/tamil-llama-3.1-8b-dpo-v1"
HF_TOKEN        = None

print(f"SFT checkpoint : {SFT_CHECKPOINT}")
print(f"Pref pairs     : {N_PREF_PAIRS}")
print(f"DPO beta       : {DPO_BETA}")

In [ ]:
# ── Load SFT model for preference pair generation ──────────────────────────────
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = SFT_CHECKPOINT,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
    token          = HF_TOKEN,
)

FastLanguageModel.for_inference(model)
print("SFT model loaded for pref pair generation.")

In [ ]:
# ── Load seed instructions ─────────────────────────────────────────────────────
# Use our existing synthetic instruction dataset as seed prompts
from datasets import load_dataset
import random

try:
    seed_ds = load_dataset("wickkiey/tamil-synthetic-instructions", split="train")
    seed_ds = seed_ds.shuffle(seed=42).select(range(min(N_PREF_PAIRS * 2, len(seed_ds))))
except Exception:
    # Fallback: load from local JSONL
    seed_ds = load_dataset("json", data_files="../data/synthetic_instructions.jsonl", split="train")

prompts = [row["instruction"] for row in seed_ds]
print(f"Seed prompts loaded: {len(prompts):,}")

In [ ]:
# ── Generate preference pairs ──────────────────────────────────────────────────
# For each instruction, generate:
#   chosen   → low temperature (T=0.3) — focused, higher quality
#   rejected → high temperature (T=1.2) — noisy, repetitive, lower quality
from tqdm import tqdm
import json

TAMIL_START, TAMIL_END = 0x0B80, 0x0BFF
def tamil_ratio(text):
    if not text: return 0.0
    return sum(1 for c in text if TAMIL_START <= ord(c) <= TAMIL_END) / len(text)

def generate_response(prompt_text: str, temperature: float) -> str:
    formatted = f"### கட்டளை:\n{prompt_text}\n\n### பதில்:\n"
    inputs    = tokenizer(formatted, return_tensors="pt", truncation=True, max_length=512).to("cuda")
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens = MAX_GEN_TOKENS,
            temperature    = temperature,
            do_sample      = True,
            pad_token_id   = tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

pref_pairs = []
PREF_FILE  = "../data/preference_pairs.jsonl"

for prompt_text in tqdm(prompts[:N_PREF_PAIRS * 2], desc="Generating preference pairs"):
    if len(pref_pairs) >= N_PREF_PAIRS:
        break

    chosen   = generate_response(prompt_text, CHOSEN_TEMP)
    rejected = generate_response(prompt_text, REJECTED_TEMP)

    # Quality gate: chosen must be better by heuristics
    chosen_score   = tamil_ratio(chosen)   * min(len(chosen), 300) / 300
    rejected_score = tamil_ratio(rejected) * min(len(rejected), 300) / 300

    # Skip if they're too similar (DPO needs clear signal)
    if abs(chosen_score - rejected_score) < 0.05:
        continue

    # Ensure chosen is actually better
    if chosen_score < rejected_score:
        chosen, rejected = rejected, chosen  # swap

    pair = {
        "prompt":   f"### கட்டளை:\n{prompt_text}\n\n### பதில்:\n",
        "chosen":   chosen,
        "rejected": rejected,
    }
    pref_pairs.append(pair)

    # Write incrementally
    with open(PREF_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(pair, ensure_ascii=False) + "\n")

print(f"\nPreference pairs generated: {len(pref_pairs):,}")

In [ ]:
# ── Prepare DPO dataset ────────────────────────────────────────────────────────
from datasets import Dataset

dpo_dataset = Dataset.from_list(pref_pairs)
dpo_dataset = dpo_dataset.shuffle(seed=42)

# Split 90/10 train/eval
split = dpo_dataset.train_test_split(test_size=0.1, seed=42)
train_dpo = split["train"]
eval_dpo  = split["test"]

print(f"DPO train: {len(train_dpo):,}  |  eval: {len(eval_dpo):,}")
print(f"\nSample:")
print(f"  Prompt  : {train_dpo[0]['prompt'][:80]}...")
print(f"  Chosen  : {train_dpo[0]['chosen'][:80]}...")
print(f"  Rejected: {train_dpo[0]['rejected'][:80]}...")

In [ ]:
# ── Reload model with LoRA for DPO training ───────────────────────────────────
# Need to re-load in training mode (not inference mode)
del model
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = SFT_CHECKPOINT,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
    token          = HF_TOKEN,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_RANK,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha          = LORA_RANK * 2,
    lora_dropout        = 0,
    bias                = "none",
    use_gradient_checkpointing = "unsloth",
    random_state        = 42,
)

print("Model ready for DPO training.")
model.print_trainable_parameters()

In [ ]:
# ── DPO Trainer ────────────────────────────────────────────────────────────────
from trl import DPOTrainer, DPOConfig
from unsloth import is_bfloat16_supported

dpo_config = DPOConfig(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = DPO_EPOCHS,
    per_device_train_batch_size = DPO_BATCH,
    per_device_eval_batch_size  = DPO_BATCH,
    gradient_accumulation_steps = DPO_GRAD_ACCUM,
    learning_rate               = DPO_LR,
    beta                        = DPO_BETA,
    fp16                        = not is_bfloat16_supported(),
    bf16                        = is_bfloat16_supported(),
    logging_steps               = 10,
    optim                       = "adamw_8bit",
    lr_scheduler_type           = "cosine",
    seed                        = 42,
    save_strategy               = "steps",
    save_steps                  = 100,
    save_total_limit            = 2,
    eval_strategy               = "steps",
    eval_steps                  = 100,
    max_length                  = MAX_SEQ_LEN,
    max_prompt_length           = 512,
    report_to                   = "none",
)

trainer = DPOTrainer(
    model         = model,
    args          = dpo_config,
    train_dataset = train_dpo,
    eval_dataset  = eval_dpo,
    tokenizer     = tokenizer,
)

print("DPO trainer initialized.")

In [ ]:
# ── Train DPO ──────────────────────────────────────────────────────────────────
trainer_stats = trainer.train()
print(f"DPO training done. Time: {trainer_stats.metrics['train_runtime']:.0f}s")

In [ ]:
# ── Compare SFT vs DPO responses ──────────────────────────────────────────────
FastLanguageModel.for_inference(model)

test_prompt = "### கட்டளை:\nதமிழ் மொழியின் தனிச்சிறப்புகள் என்ன?\n\n### பதில்:\n"
inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

with torch.inference_mode():
    output = model.generate(
        **inputs, max_new_tokens=200, temperature=0.7,
        do_sample=True, pad_token_id=tokenizer.eos_token_id,
    )

print("DPO model response:")
print(tokenizer.decode(output[0], skip_special_tokens=True))

In [ ]:
# ── Save & push ────────────────────────────────────────────────────────────────
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

try:
    from google.colab import drive
    model.save_pretrained(f"/content/drive/MyDrive/Tamil-LLM/{OUTPUT_DIR}")
except ImportError:
    pass

model.push_to_hub(HF_REPO, token=HF_TOKEN,
    commit_message="DPO v1 — preference optimization on SFT checkpoint")
tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
print(f"Pushed: https://huggingface.co/{HF_REPO}")

## Next Steps

1. Run `evaluation/03_benchmark_eval.ipynb` — compare DPO model against SFT and Sarvam-1 baselines
2. If DPO improvements plateau after multiple rounds, proceed to `finetuning/04_rlhf_ppo.ipynb` for RLHF
3. Otherwise, iterate: generate more preference pairs, re-run DPO with fresh data